In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import mlflow
import mlflow.pytorch
import torchvision
import torchvision.transforms as transforms


In [11]:

# Define a simple neural network
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.float()
        # print(f'inside model {x.shape}')
        x = x.view(-1, 28 * 28)
        # print(f'after view {x.shape}')
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [12]:


# Load dataset
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.MNIST(root="./mnist", train=True, transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

In [13]:
model = SimpleNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("test1")

with mlflow.start_run():
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("epochs", 5)

    for epoch in range(5):
        total_loss = 0
        for images, labels in train_loader:
            optimizer.zero_grad()
            # print(f'inside training {images.shape}')
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/5], Loss: {avg_loss:.4f}")
        mlflow.log_metric("loss", avg_loss, step=epoch)

    # Log and register model
    mlflow.pytorch.log_model(model, "model", registered_model_name="mnist_model")

Epoch [1/5], Loss: 0.2948
Epoch [2/5], Loss: 0.1290
Epoch [3/5], Loss: 0.0878
Epoch [4/5], Loss: 0.0655
Epoch [5/5], Loss: 0.0511


2025/04/04 01:49:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'mnist_model' already exists. Creating a new version of this model...
2025/04/04 01:49:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: mnist_model, version 2


🏃 View run mercurial-donkey-192 at: http://localhost:5000/#/experiments/770654020561177755/runs/46e61e2448084de49ad814d0a18bf2ae
🧪 View experiment at: http://localhost:5000/#/experiments/770654020561177755


Created version '2' of model 'mnist_model'.


In [10]:
torch.rand(32,1, 28*28).view(-1,28*28).shape

torch.Size([32, 784])

In [15]:
import requests
import torch

url = "http://localhost:5001/invocations"

# Create a random tensor matching MNIST input shape (batch_size=1, 28*28)
input_tensor = torch.rand(32, 1, 28 , 28)

# Convert to list
data = {"inputs": input_tensor.tolist()}

# Send request
response = requests.post(url, json=data)

# Print prediction
print(response.json())


{'predictions': [[-10.29250717163086, -20.507343292236328, 6.661235332489014, 1.3624478578567505, -42.71775817871094, 6.29326057434082, -0.3324410319328308, -1.4703677892684937, -7.143215656280518, -20.601659774780273], [-5.935019493103027, -19.96959686279297, 6.1248393058776855, 7.0228424072265625, -45.08654022216797, 8.572267532348633, -8.57303524017334, 3.0423176288604736, -10.19649887084961, -18.868581771850586], [-5.222397327423096, -20.365951538085938, 4.561220645904541, 4.664946556091309, -40.48066711425781, 6.392124176025391, -6.974786281585693, -0.37819987535476685, -6.282113075256348, -14.969465255737305], [-3.5336930751800537, -23.396644592285156, 6.8793463706970215, 9.405579566955566, -45.74473571777344, 7.7614898681640625, -7.536750316619873, -7.731308460235596, -11.027837753295898, -16.765283584594727], [-4.8482842445373535, -22.911540985107422, 6.854462623596191, 6.686636924743652, -45.84922790527344, 8.077987670898438, -13.297436714172363, 1.67432701587677, -5.965930938

In [7]:
torch.rand(1,768).view(-1,768).shape

torch.Size([1, 768])